In [ ]:
import math
import copy
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from collections.abc import Callable
from typing import Any, List, Tuple, Any, Dict
from itertools import count
from importnb import Notebook

with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.PrefetchScheduler import PrefetchScheduler

import os
import sys

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

import Common.config as config
import Common.datatypes as datatypes
import Common.utils as utils

import importlib

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(utils)

ModuleNotFoundError: No module named 'Labs'

In [ ]:
from numpy import info


class EnvWrapper(gym.Env):
    """
    Simple wrapper that delegates all calls to an inner env.
    Subclass this to create your own wrappers.
    """

    metadata = {"render.modes": []}

    def __init__(
        self, 
        cfg: Any,
        n: int,
        m: int,
        n_layers: int,
        lam: float,
        theta: float,
        users_env: UserRequestEvents = None,
        du_caches: list = None,
        mec_cache: CacheEngineEnv = None,
        latency_model: MultiDULatencyModel = None,
        prefetch_fn: Callable = None,
        reward_fn: Callable = None,
        max_steps: int = 10000,
        *,
        step_duration_s: float = 1.0,
        debugger=None
    ):
        super().__init__()

        self.cfg = cfg 
        self.step_count = 0
        self.step_duration_s = step_duration_s
        self.max_steps = max_steps

        self.n = n  # number of tiles per row/column
        self.m = m  # number of tiles per row/column
        self.n_layers = n_layers  # number of layers (base + enhancement)

        self.gain_if_prefetched = 1.0
        self.loss_if_not_prefetched = -1.0

        self.k_meta = 600.0  # Scales probability delta up to PSNR 30 bound
        self.k_ctrl = 250.0  # Scales probability delta up to PSNR 2.5 bound

        self.theta = theta
        self.lam = lam

        self.users_env = users_env
        self.du_caches = du_caches or []
        self.mec_cache = mec_cache
        self.latency_model = latency_model

        self.prefetch_fn = prefetch_fn or (lambda cache, action: cache.drl_prefetching(action))
        self.reward_fn = reward_fn or (lambda info: info.get('reward_per_user', {}).get(info.get('current_user', -1), 0.0))

        # History holders
        self.users_reward: Dict[int, list] = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr: Dict[int, list] = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user: Dict[int, int] = {
            u: 0 for u in range(self.users_env.n_users)
        }
        
        self.scheduler = PrefetchScheduler(
            R_M_D=self.latency_model.R_M_D,
            R_C_M=self.latency_model.R_C_M,
            U=self.latency_model.max_U,
            step_duration_s=self.step_duration_s
        )
        
        self.debugger = debugger

    # ─────────────────────────────────────────────────────────────────────────
    # Internal Helpers
    # ─────────────────────────────────────────────────────────────────────────
    def _missing_items(self, req: Dict[str, Any]) -> list[int]:
        video = req["video"]
        viewport = req["viewport"]

        cache = self.mec_cache.policy.cache

        has_video = (video, -1) in cache
        vp = [(video, tile) for tile in viewport]
        
        return [int(not has_video)] + [int(v not in cache) for v in vp]

    def _process_prefetch_actions(
        self, action, req, du_plan, mec_plan
    ):
        if req is None or action is None:
            return False, False, None, []

        video = req["video"]
        viewport = req["viewport"]

        trans_enh = False
        trans_base = False

        intended_enh = []
        intended_base = None

        backhaul_usage = 0
        missing = self._missing_items(req)

        for i, is_missing in enumerate(missing):
            if not is_missing:
                continue

            video_cache_idx = self.mec_cache.get_video_cache_idx(video)
            if i > 0 and video_cache_idx == -1:
                break

            message = {
                "video": video,
                "tiles": [] if i == 0 else [viewport[i - 1]], 
                "base_req_init": (i == 0), 
                "action_idx": action[i],
            }
            self.prefetch_fn(self.mec_cache, message)

            if i == 0:
                trans_base = True

                if action[i] != 0 and video_cache_idx == -1:
                    backhaul_usage += 12 * self.mec_cache.tile_size_bytes[0]
                    intended_base = video
            else:
                trans_enh = True

                if action[i] != 0 and (video, viewport[i - 1]) not in self.mec_cache.policy.cache:
                    backhaul_usage += self.mec_cache.tile_size_bytes[1]
                    intended_enh.append(viewport[i - 1])

        self.debugger.log("backhaul_usage", backhaul_usage)

        ### Logging for debugging and analysis ###
        if trans_base:
            self.debugger.log("base_action", action[0])
        self.debugger.log("base_layer_miss", missing[0])

        if trans_enh:
            for i, act in enumerate(action[1:]):
                self.debugger.log(f"enh_{i}_action", act)

        for i, is_missing in enumerate(missing[1:], start=1):
            self.debugger.log(f"enh_layer_missing_{i}", is_missing)

        for vp in viewport:
            self.debugger.log("tile_viewport", vp)

        return trans_base, trans_enh, intended_base, intended_enh

    def _compute_cache_hits(self, req):
        video = req["video"]
        viewport = req["viewport"]

        cache = self.mec_cache.policy.cache

        base_hit = 1 if (video, -1) in cache else 0

        vp = [(video, tile) for tile in viewport]
        enh_hit = [int(v in cache) for v in vp]
    
        return dict(
            base_layer_hits=12 * base_hit,
            base_layer_misses=12 * (1 - base_hit),
            enh_layer_hits=sum(enh_hit),
            enh_layer_misses=4 - sum(enh_hit)
        )

    def _expected_cpt_reward(self, req, video_p):

        video_cache_index = self.mec_cache.policy.cache
        
        
        prefetch = 1 if (req["video"], -1) in video_cache_index else 0

        w = self._prelec_w(video_p, theta=self.theta)
        
        if prefetch == 1:
            expected_outcome = self.gain_if_prefetched
        else:
            expected_outcome = self.loss_if_not_prefetched
        
        v = self._cpt_value(expected_outcome, lam=self.lam)

        expected_reward = w * v
        return expected_reward
    
    def _expected_cpt_reward_enh(self, is_missing, video_p):
        
        prefetch = 1 if is_missing == 0 else 0

        w = self._prelec_w(video_p, theta=self.theta)
        
        if prefetch == 1:
            expected_outcome = self.gain_if_prefetched
        else:
            expected_outcome = self.loss_if_not_prefetched
        
        v = self._cpt_value(expected_outcome, lam=self.lam)

        expected_reward = w * v
        return expected_reward

    def _prelec_w(self, p: float, theta: float = 0.65) -> float:
        if p <= 0.0: return 0.0
        if p >= 1.0: return 1.0
        return math.exp(-((-math.log(p)) ** theta))

    def _cpt_value(
        self,
        x: float, 
        lam: float = 1.5
    ) -> float:
        if x >= 0:
            return x
        else:
            return -lam * (-x)

    def _serve_with_latency_budget(
        self,
        req: Dict[str, Any],
        threshold_s: float = 1.0,
        t_base_cache: float = 0.16666 * 2,
        t_base_backhaul: float = 0.5 * 2,
        t_enh_cache: float = 0.16666,
        t_enh_backhaul: float = 0.5,
    ):
        """
        Parameters:
            req: Dict[str, Any]
            Request dict with keys:
            - "video": video id
            - "viewport": list of requested enhancement tile ids.
            threshold_s: float, default=1.0
            Per-request latency budget in seconds.
            t_base_cache: float, default=0.16666 * 2
            Time to deliver the base layer when it is already in MEC cache.
            t_base_backhaul: float, default=0.5 * 2
            Time to deliver the base layer via backhaul when it is missing in cache.
            t_enh_cache: float, default=0.16666
            Time to deliver one enhancement tile from MEC cache.
            t_enh_backhaul: float, default=0.5
            Time to deliver one enhancement tile via backhaul.

        Returns:
            base_delivered: int (0/1)
            enh_delivered: list[int] length=viewport
            total_time: float
            enh_from_cache: int
            enh_from_backhaul: int
        """

        video = req["video"]
        viewport = req["viewport"]
        cache = self.mec_cache.policy.cache

        # ---- Base first ----
        base_in_cache = (video, -1) in cache
        base_time = t_base_cache if base_in_cache else t_base_backhaul

        if base_time > threshold_s:
            return 0, [0] * len(viewport), threshold_s, 0, 0

        remaining = threshold_s - base_time
        enh_delivered = [0] * len(viewport)
        enh_from_cache = 0
        enh_from_backhaul = 0

        # 1) Cache tiles first
        for i, vp in enumerate(viewport):
            tile = viewport[i]
            if (video, tile) in cache and remaining >= t_enh_cache:
                enh_delivered[i] = 1
                remaining -= t_enh_cache
                enh_from_cache += 1

        # 2) Backhaul for misses (only if action asks for it: score > 0)
        for i, vp in enumerate(viewport):
            tile = viewport[i]
            if enh_delivered[i] == 1:
                continue
            if remaining >= t_enh_backhaul:
                enh_delivered[i] = 1
                remaining -= t_enh_backhaul
                enh_from_backhaul += 1

        total_time = threshold_s - remaining
        return 1, enh_delivered, total_time, enh_from_cache, enh_from_backhaul

    def _update_cache_hit_history(self, video, viewport):
        hit_vid = 1 if (video, -1) in self.mec_cache.policy.cache else 0
        hits_vp = [
            1 if (video, tile) in self.mec_cache.policy.cache else 0 
            for tile in viewport
        ]

        self.all_video_hist.append(hit_vid)
        self.all_vp_list.append(hits_vp)

    def _compute_reward_layer_0(self, window_size=100):
        return 30.0 * sum(self.all_video_hist[-window_size:]) / window_size
    
    def _compute_reward_layer_1(self, window_size=100):
        recent_vp = self.all_vp_list[-window_size:]
        if not recent_vp:
            return 0.0

        enh_hits = sum(sum(vp) for vp in recent_vp)
        total_enh = len(recent_vp) * 4  # 4 tiles per viewport

        return 10.0 * enh_hits / (total_enh if total_enh > 0 else 1)

    # ─────────────────────────────────────────────────────────────────────────
    # Gym Environment API
    # ─────────────────────────────────────────────────────────────────────────
    def sample_action(self) -> Tuple[Dict[str, Any], int]:
        """Sample random action with center viewport."""
        tiles = np.zeros(self.n * self.m, dtype=int)
        c = self.n // 2
        
        if self.n % 2 == 1:
            center_idx = c * self.n + c
            tiles[center_idx] = 1
        else:
            for x, y in [(c-1, c-1), (c-1, c), (c, c-1), (c, c)]:
                tiles[y * self.n + x] = 1

        return {
            'video': np.random.randint(0, self.users_env.n_videos),
            'gop': np.random.randint(0, self.users_env.n_gops),
            'tiles': tiles.tolist()
        }, c * self.n + c

    def step(
        self, 
        action: List, 
        req: list[Dict[str, Any]],
        net_adapter: Any,
    ) -> Tuple[dict, float, bool, dict]:
        info = {}

        du_plan = self.du_caches if len(self.du_caches) > 0 else None
        mec_plan = self.mec_cache.get_cache_bitmap() if self.mec_cache else None

        prefetch_base, prefetch_enh, intended_base, intended_enh = self._process_prefetch_actions(
            action, req, du_plan, mec_plan
        )

        # -------------------------------------------------------
        # NEW: Evaluate Prefetching Time Constraints
        # -------------------------------------------------------

        
        # missing = self._missing_items(req)
        
        # intended_base = req["video"] if missing[0] == 0 else None
        # intended_enh = [
        #     req["viewport"][i - 1] for i, is_missing in enumerate(missing[1:], start=1) if is_missing == 0
        # ]

        # current_time = self.users_env.get_current_time() 
        # bandwidth = net_adapter.get_available_bandwidth()
        # latency = net_adapter.get_latency()
        
        # prefetch_base = []
        # prefetch_enh = []

        # # Evaluate Base Focus
        # if intended_base:
        #     size = self.mec_cache.get_item_size(intended_base)
        #     fetch_time = (size / bandwidth) + latency
        #     deadline = self.users_env.get_request_time(intended_base) # When the user needs it
            
        #     if current_time + fetch_time <= deadline:
        #         prefetch_base.append(intended_base)
        #         # (Optional) Update bandwidth here if downloading in parallel
        #     else:
        #         info[f"failed_prefetch_{intended_base}"] = "Missed Deadline"

        # # Evaluate Enhancement Focus
        # for item in intended_enh:
        #     size = self.mec_cache.get_item_size(item)
        #     fetch_time = (size / bandwidth) + latency
        #     deadline = self.users_env.get_request_time(item)
            
        #     if current_time + fetch_time <= deadline:
        #         prefetch_enh.append(item)
        #     else:
        #         info[f"failed_prefetch_{item}"] = "Missed Deadline"

        # # (Important) Tell the cache to ONLY store the successful ones
        # self.mec_cache.execute_valid_prefetches(prefetch_base, prefetch_enh)
        
        # -------------------------------------------------------
        # Continue with standard step logic...
        # -------------------------------------------------------
        nxt_req = self.users_env.get_next_request(du_plan, mec_plan)
        video = nxt_req["video"]
        viewport = nxt_req["viewport"]

        info["prefetch_base"] = prefetch_base
        info["prefetch_enh"] = prefetch_enh

        info["video_cache_idx"] = self.mec_cache.get_video_cache_idx(video)

        # -------------------------------------------------------
        # 2. Remaining users requests
        # -------------------------------------------------------
        info["user_request"] = nxt_req

        # -------------------------------------------------------
        # 3. Cache stats (HIT / MISS)
        # -------------------------------------------------------
        info.update(self._compute_cache_hits(nxt_req))

        # -------------------------------------------------------
        # 4. CPT reward components
        # -------------------------------------------------------
        net_adapter.features.update_history(video, viewport)

        self._update_cache_hit_history(video, viewport)
        reward_0 = self._compute_reward_layer_0(window_size=1)
        reward_1 = self._compute_reward_layer_1(window_size=1)

        # 4.1 CPT reward for base layer prefetching
        # short_total = sum(net_adapter.features.video_freq_short.values())
        long_total = sum(net_adapter.features.video_freq_long.values())

        # p_short = net_adapter.features.video_freq_short.get(video, 0.0) / (short_total if short_total > 0 else 1.0)
        p_long = net_adapter.features.video_freq_long.get(video, 0.0) / (long_total if long_total > 0 else 1.0)
                
        video_p = p_long
        # video_p = p_short * 0.3 + p_long * 0.7
        reward_0 = self._expected_cpt_reward(nxt_req, video_p)

        # # 4.2 CPT reward for enhancement layer prefetching
        # long_total_enh = sum(net_adapter.features.tile_freq_long.values())
        # short_total_enh = sum(net_adapter.features.tile_freq_short.values())

        # tile_ps = []
        # for tile in viewport:
        #     p_short_enh = net_adapter.features.tile_freq_short.get((video, tile), 0.0) / (short_total_enh if short_total_enh > 0 else 1.0)
        #     p_long_enh = net_adapter.features.tile_freq_long.get((video, tile), 0.0) / (long_total_enh if long_total_enh > 0 else 1.0)
        #     tile_p = p_short_enh * 0.3 + p_long_enh * 0.7
        #     tile_ps.append(tile_p)

        # is_missing = self._missing_items(nxt_req)[1:]
        # reward_1 = sum(self._expected_cpt_reward_enh(is_missing, tile_p) for tile_p, is_missing in zip(tile_ps, is_missing))

        info["reward_layer_0"] = reward_0
        info["reward_layer_1"] = reward_1

        # print(reward_0, reward_1)

        # -------------------------------------------------------
        # 4.3 LATENCY
        # -------------------------------------------------------
        base_ok, enh_ok, total_time, n_enh_cache, n_enh_backhaul = self._serve_with_latency_budget(
            req=nxt_req, threshold_s=1.0
        )

        is_missing = [0 if base_ok else 1] + [0 if x == 1 else 1 for x in enh_ok]

        # -------------------------------------------------------
        # 4.3 PSNR 
        # -------------------------------------------------------
        psnr_0 = 30.0 if not is_missing[0] else 0.0
        psnr_1 = sum(10.0 for miss in is_missing[1:] if not miss)
        psnr_total = psnr_0 + (psnr_1 / 4.0)

        info["psnr"] = psnr_total

        # print(psnr_total, reward_0 + reward_1)
        # -------------------------------------------------------
        # 5. Store transitions and train
        # -------------------------------------------------------

        done = self.users_env.all_users_done()
        self.step_count += 1

        # print(f"Reward: {reward_0}, Shaping: {shaping_signal_0:.2f}, Total: {shaped_reward_0:.2f} | Reward: {reward_1}, Shaping: {shaping_signal_1:.2f}, Total: {shaped_reward_1:.2f}")
        # print(f"Step {self.step_count} | U: {nxt_req['u']} | V: {video} | GOP: {nxt_req['gop']} | RL0: {reward_0:.2f} | RL1: {reward_1:.2f} | BH: {info['base_layer_hits']} | EH: {info['enh_layer_hits']} | BM: {info['base_layer_misses']} | EM: {info['enh_layer_misses']}")

        # info["reward_layer_0"] = reward_0 = psnr_0
        # info["reward_layer_1"] = reward_1 = psnr_1 / 4.0

        self.debugger.log('reward', reward_0 + reward_1)
        self.debugger.log('psnr', psnr_total)

        return None, reward_0 + reward_1, done, info


    # ---------------------------------------------------------
    #  LATENCY + BANDWIDTH COST
    # ---------------------------------------------------------
    def compute_latency_and_bw(self, reqs: list[Dict[str, Any]]) -> Tuple[Dict[int, float], Dict[int, float]]:
        """Compute latency and bandwidth cost per user based on cache hits/misses."""
        latency_per_user: Dict[int, float] = {u: 0.0 for u in range(self.users_env.n_users)}
        bw_cost_per_user: Dict[int, float] = {u: 0.0 for u in range(self.users_env.n_users)}

        for req in reqs:
            u = req["u"]

            for tile in req["tiles"]:
                layer = tile["layer"]
                size = self.mec_cache.tile_size_bytes[layer]
                du_hit = tile["events"]["alpha_p_u"]
                mec_hit = tile["events"]["alpha_M_u"]

                # Classify event location
                per_byte_latency = (
                    self.latency_model.TD_U if du_hit == 1
                    else self.latency_model.R_M_D if mec_hit == 1
                    else self.latency_model.R_C_M
                )

                latency_per_user[u] += per_byte_latency * size
                bw_cost_per_user[u] += size

        return latency_per_user, bw_cost_per_user

    # ---------------------------------------------------------
    #  PSNR MODEL
    # ---------------------------------------------------------
    def compute_psnr(self, reqs: list[Dict[str, Any]]) -> Dict[int, float]:
        """Compute PSNR per user based on layer hits."""
        psnr_per_user: Dict[int, float] = {}
        
        for req in reqs:
            u = req["u"]
            base_sum = sum(
                30.0 for tile in req["tiles"]
                if tile["layer"] == 0 and tile["events"]["alpha_M_u"] == 1
            )
            enh_sum = sum(
                10.0 for tile in req["tiles"]
                if tile["layer"] == 1 and tile["events"]["alpha_M_u"] == 1
            )
            psnr_per_user[u] = (base_sum / 12.0) + (enh_sum / 4.0)

        return psnr_per_user

    # ---------------------------------------------------------
    # REWARD FUNCTION
    # ---------------------------------------------------------
    def compute_reward(self, psnr_per_user: Dict[int, float]) -> Dict[int, float]:
        """Map PSNR to reward."""
        return psnr_per_user

    # ---------------------------------------------------------
    #  HIT / MISS STATS
    # ---------------------------------------------------------
    # def compute_cache_stats(self, reqs: list[Dict[str, Any]]) -> Dict[str, int]:
    #     """Compute cache hit/miss statistics by layer."""
    #     stats = {"base_layer_hits": 0, "enh_layer_hits": 0, "base_layer_misses": 0, "enh_layer_misses": 0}
        
    #     for req in reqs:
    #         for tile in req["tiles"]:
    #             layer = tile["layer"]
    #             hit = int(tile["events"]["alpha_p_u"] or tile["events"]["alpha_M_u"])
                
    #             match layer:
    #                 case 0:
    #                     stats["base_layer_hits"] += hit
    #                     stats["base_layer_misses"] += (1 - hit)
    #                 case _:
    #                     stats["enh_layer_hits"] += hit
    #                     stats["enh_layer_misses"] += (1 - hit)

    #     return stats

    # ---------------------------------------------------------
    # WARM-UP PHASE
    # ---------------------------------------------------------
    def warmup_phase(self, net_adapter, num_steps=100):
        """Warm-up phase to initialize cache with some videos."""

        base_idx_counter = 0

        for _ in range(num_steps):  # Arbitrary number of warm-up steps
            req = self.users_env.get_next_request(None, None)
            video = req["video"]
            viewport = req["viewport"]

            vid_idx = self.mec_cache.get_video_cache_idx(video)
            if vid_idx == -1:
                action_idx = base_idx_counter 
                base_idx_counter = (base_idx_counter + 1) % (self.cfg.cache_size * (self.cfg.viewport + 1))
            else:
                action_idx = vid_idx

            action = {
                "video": video,
                "tiles": [],
                "base_req_init": True,
                "action_idx": action_idx
            }
            self.prefetch_fn(self.mec_cache, action)

            for idx, tile in enumerate(viewport):

                has_tile = (video, tile) in self.mec_cache.policy.cache

                if has_tile:
                    continue

                tile_action = {
                    "video": video,
                    "tiles": [tile],
                    "base_req_init": False,
                    "action_idx": base_idx_counter
                }
                self.prefetch_fn(self.mec_cache, tile_action)

                base_idx_counter = (base_idx_counter + 1) % (self.cfg.cache_size * (self.cfg.viewport + 1))

            net_adapter.features.update_history(video, viewport)

        ### Final cache state after warm-up ###
        # req = self.users_env.get_next_request(None, None)
        # video = req["video"]
        # viewport = req["viewport"]

        # net_adapter.features.update_history(video, viewport)
        # net_adapter.features.update_ch_history(video, viewport)

    # ---------------------------------------------------------
    # RESET
    # ---------------------------------------------------------
    def reset(self, **kwargs) -> Tuple[None, Dict[str, Any]]:
        """Reset environment to initial state."""
        self.step_count = 0
        self.nxt_req = None

        self.users_reward = {u: [] for u in range(self.users_env.n_users)}
        self.users_psnr = {u: [] for u in range(self.users_env.n_users)}
        self.total_gop_requests_per_user = {u: 0 for u in range(self.users_env.n_users)}

        _, info_users = self.users_env.reset(**kwargs)
        info_cache_mec = self.mec_cache.reset(**kwargs)[1] if self.mec_cache else {}

        info_cache = {
            **info_cache_mec,
            **info_users
        }

        self.scheduler.now_s = 0.0
        self.scheduler.availability = {}

        self.all_video_hist = []
        self.all_vp_list = []

        return None, info_cache

In [ ]:
def getTiles(step: int, user: int, users_viewport_tiles: Dict[int, list], n: int) -> np.ndarray:
    """Convert viewport tiles for user at step to binary mask."""
    mask = np.zeros(n * n, dtype=int)
    
    for tx, ty in users_viewport_tiles[user][step]:
        if 0 <= tx < n and 0 <= ty < n:
            mask[ty * n + tx] = 1
    
    return mask

In [ ]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserRequestEvents(
        n_nodes=n_nodes,
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    # du_caches = [
    #     CacheEngineEnv(
    #         n_tiles=n*n,
    #         n_videos=n_videos,
    #         cache_capacity=max_capacity
    #     ) for _ in range(n_nodes)  # Number of DUs = n_nodes
    # ]
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward_0, done, info = env.step(actions)

        total_reward += float(reward_0)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward_0:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )

        if done: 
            break

In [ ]:
if __name__ == "__main__":
    steps = range(1, len(results) + 1)
    cache_hits_series = [r["cache_hits"] for r in results]
    cache_misses_series = [r["cache_misses"] for r in results]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)

    # Rewards with moving average
    axes[0].plot(steps, [r["total_reward"] for r in results], label="Step reward", alpha=0.7, color="blue")
    w = max(1, min(20, len(results) // 10))
    if w > 1:
        ma = [sum([r["total_reward"] for r in results][i - w:i]) / w for i in range(w, len(results) + 1)]
        axes[0].plot(range(w, len(results) + 1), ma, label=f"Moving avg (w={w})", color="orange")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Total reward")
    axes[0].set_title("Training Rewards")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Cache hits
    axes[1].plot(steps, cache_hits_series, label="Cache hits", color="green", alpha=0.8)
    axes[1].set_title("Cache Hits per Step")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Hits")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    # Cache misses
    axes[2].plot(steps, cache_misses_series, label="Cache misses", color="red", alpha=0.8)
    axes[2].set_title("Cache Misses per Step")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Misses")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserTileRequestEvents(
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward_0, done, info = env.step(actions)

        total_reward += float(reward_0)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward_0:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )
        
        if done: 
            break